# 1. Cache (캐시)

- 캐싱(caching): 특정 함수의 실행 결과(예: 데이터 로딩, API 호출 등)를 저장해두고, 같은 입력으로 함수가 다시 호출되면 저장된 결과를 재사용하는 방식 
    - 함수 실행이 느리거나 비용이 클 때, 결과를 저장해두면 함수 재실행 과정 없이 저장해둔 결괏값만 들고옴으로써 앱이 훨씬 빨라짐 
    - Streamlit 처럼 매 번 스크립트 전체가 재실행되는 구조에서는 캐싱이 필수 
    - 시간과 비용이 많이 들거나 자주 엑세스 하는 계산 결과를 저장해둠으로써 웹이나 프로그램이 실행될 때 마다 다시 계산할 필요가 없도록 효율화 하여 프로그램의 성능을 높이는 데 도움을 주는 메커니즘 (**임시 저장소의 개념**)
- Streamlit은 웹에서 데이터를 불러오거나, 대용량 데이터셋을 처리하거나, 비용이 많이 드는 연산을 수행할 때에도 앱의 성능을 유지할 수 있도록 강력한 캐시 기본 기능을 **데코레이터**로 제공 
    - @st.cache_data: 데이터 결과용 캐시 
        - 용도: DataFrame, list, dict, str 등 결과값 중심인 함수에 사용 (예: CSV 로딩, 웹 API 요청, 계산 함수 결과 등)
        - 특징: 입력값이 같으면 **이전 결과를 재사용** 하며 결과는 pickle로 직렬화되어 저장되고 캐시된 결과는 메모리에 저장 됨
            * pickle: 파이썬 객체를 저장하거나 전송할 수 있도록 변환하는 모듈로, 복잡한 파이썬 객체를 바이트 형태로 바꿔서 파일로 저장하거나 메모리에 보관할 수 있게 해줌 
    - @st.cache_resource: 리소스 객체용 캐시 
        - 용도: 모델 DB 연결, 네트워크 세션 등 리소스 객체 중심 함수에 사용 (예: 인공지능 모델 로드, DB 커넥션, 세션 연결 등)
        - 특징: **객체 자체**를 캐시하므로 상태 유지에 적합하며, 결과가 객체, 인스턴스일 때 사용해야 안전 

- 아래와 같은 비효율적인 상황을 피하기 위해 사용
    - 시간이 많이 소요되는 함수가 불필요하게 반복되는 상황
    - 한 번 만들어두면 되는 객체가 새로고침 할 때 마다 재생성되는 상황

In [15]:
%%writefile my_web/07_01_status.py
import streamlit as st
import time
import pandas as pd
import numpy as np

@st.cache_data
def load_df():
    time.sleep(2) 
    
    df = pd.DataFrame({
        'co1': np.random.randn(100),
        'co2': np.random.randn(100)
    })
    
    return df


if st.button('데이터불러오기', key='load_df'):
    df = load_df()
    st.write('데이터 로드 완료')
    st.write(df)
    
    
st.markdown("---")
st.subheader(" @cache_data 예제2")

@st.cache_data
def get_data(age):
    my_dict = {'name':['최희윤'], 'age':[age], 'gender':['female']}
    df = pd.DataFrame(my_dict)
    time.sleep(3)
    
    return df

# 나이를 전달할 input widget 설정
age = st.number_input('나이를 입력해주세요:', min_value=0, max_value=100, value=50, step=1)

if st.button('데이터 불러오기', key='load_df2'):
    df = get_data(age)
    st.success('데이터 로드 완료')
    st.write(df)
    
    
st.markdown('---')

st.subheader('모든 캐시 삭제')

if st.button('모든 캐시 삭제', key='cache_clear'):
    st.cache_data.clear()
    st.success('모든 캐시가 삭제')
    
# 삭제할 함수 선택

selected_function = st.selectbox('캐시를 삭제할 함수를 선택하세요:', options=['load_df()', 'get_data()'], placeholder='Choose an Option')
if st.button('캐시 삭제'):
    if selected_function == 'load_df()':
        load_df.clear()
    else:
        get_data.clear()
        
    st.success(f'{selected_function}의 캐시 삭제 완료')


Overwriting my_web/07_01_status.py


# 2.Form(폼)
- "제출"버튼과 함께 **요소들을 일괄 처리**하는 컨테이너 
- 다른 요소들과 위젯들을 시각적으로 그룹화하고, "제출"버튼을 포함하는 컨테이너 
    - "제출"버튼이 눌리면, 폼 내의 모든 위젯값들이 일괄적으로 Streamlit에 전송 됨 
- 요소들을 폼 객체에 추가하려면 with 구문을 사용하거나, 폼에 직접 메서드를 호출할 수 있음 
- 제약사항
    - 모든 폼에는 st.form_submit_button이 포함되어야 함
    - st.button 및 st.download_button은 폼에 추가할 수 없음 
    - 폼은 앱 어디든 위치 가능(사이드바, 컬럼 등)하지만, 다른 폼 안에 중첩될 수는 없음 
    - 폼 내에서 콜백함수를 가질 수 있는 유일한 위젯은 st.from_submit_button 
        * 콜백함수: 어떤 동작이 일어났을 대 자동으로 호출되는 함수 

In [21]:
%%writefile my_web/07_2_status.py
import streamlit as st
import numpy as np
import pandas as pd
import time 

st.title('Form 예제')

# 함수 생성
@st.cache_data
def get_data(name, age, gender):
    my_dict = {'name':[name], 'age':[age], 'gender':[gender]}
    df = pd.DataFrame(my_dict)
    time.sleep(3)
    
    return df
    
st.markdown('---')


st.header('폼 미적용')

name1 = st.text_input('이름을 입력하세요')
age1 = st.number_input('나이를 입력하세요', min_value=0, max_value=100, value=50, step=1)
gender1 = st.radio('성별은 선택하세요', ['male', 'female'])

st.write(get_data(name1, age1, gender1))

with st.form(key='form1', clear_on_submit=True):
    name2 = st.text_input('이름을 입력하세요')
    age2 = st.number_input('나이를 입력하세요')
    gender2 = st.radio('성별을 선택하세요', ['male', 'female'])
    
    if st.form_submit_button('제출하기'):
        st.write(get_data(name2, age2, gender2))


Overwriting my_web/07_2_status.py


# 3. Session

- session이란? 
    - 웹은 기본적으로 무상태(stateless)임 
        - 즉, HTTP 요청 하나하나는 독립적으로 처리되며, 서버는 이전 요청에 대한 정보를 기본적으로 기억하지 않음 
    - 하지만, 웹 어플리케이션에서는 로그인 상태 유지, 장바구니 유지 등 사용자별 상태 정보 유지가 필수 
    - 이를 위해 사용하는 것이 session 
    - 예:
        - 서버가 사용자별 고유한 세션 ID를 발급하고, 해당 ID에 연결된 정보를 메모리나 데이터베이스 저장
        - 클라이언트는 세션 ID를 쿠키 등을 통해 보관하고, 이후 요청 시 서버에 전달
        - 대표적인 사용 예: 로그인 상태 유지, 장바구니 정보 저장 등 
    
- session_state란?
    - st.session_state는 사용자별로 유지되는 Python 딕셔너리처럼 동작하는 상태 저장 공간 
    - 페이지 리렌더링(버튼 클릭, 슬라이더 이동 등)이 되어도 상태가 유지됨 
        - 랜더링: 컴퓨터에서 어떤 정보를 눈에 보이는 형태(화면, 이미지, 그래프 등)로 변환해서 보여주는 과정으로 HTML, CSS 코드가 실제로 웹페이지로 보이게 만드는 과정을 의미
    - 기본적으로 메모리에 저장됨(서버 재시작 시 초기화됨)
    - Streamlit의 다양한 동작을 기능하는 함수의 매개변수 중 key가 명시되면 session_state에 자동으로 해당 key가 업데이트 됨 
        - 하지만, 해당 함수들이 form 내에 위치할 경우 key를 명시하여도 session_state에 자동으로 업데이트 되지 않음 


In [29]:
%%writefile my_web/07_03.status.py
import streamlit as st

st.title('Seession 예제')

""

st.write('### 현재 세션 상태 session_state 상태 확인')
st.write(st.session_state)



""

st.write('### 현재 세션 상태 session_state에 cnt 추가')

if 'cnt' not in st.session_state:
    st.session_state['cnt'] = 0
st.write(st.session_state)
    
st.write('button 클릭 시 session 상태가 어떻게 바뀌는지 확인')

col1, col2, col3 = st.columns(3)

if col1.button('1 증가', key='btn_1'):
    st.session_state['cnt'] += 1
    
if col2.button('1 감소', key='btn_2'):
    st.session_state['cnt'] -= 1

if col3.button('초기화', key='btn_3'):
    st.session_state['cnt'] = 0 
    
    
st.write(f'현재 cnt: {st.session_state["cnt"]}')
st.write(st.session_state)

# form을 생성하여 값을 제출할 땐 session 상태가 어떻게 바뀌는지 확인
st.write('button 클릭 시 session 상태가 어떻게 바뀌는지 확인')

st.write('제출을 누르면 초기화 되는 경우')

if st.button('폼 열기', key='btn_4'):
    with st.form(key='form1', clear_on_submit=True):
        name1 = st.text_input('이름을 입력하세요', key='text_input1')
        age1 = st.number_input('나이를 입력하세요', min_value=0, max_value=100, value=50, step=1, key='num_input1')
        gender1 = st.radio('성별은 선택하세요', ['male', 'female'], key='radio1')
    
        
        if st.form_submit_button('제출'):
            st.write(f'제출 결과: 이름:{name1}, 나이:{age1}, gender:{gender1}')
            
            

st.write('제출을 누르면 초기회 안 되는 경우')

def doc_clicked():
    return st.session_state.get('clicked', False)

if doc_clicked():
    with st.form(key='form2', clear_on_submit=False):
        name2 = st.text_input('이름을 입력하세요', key='text_input2')
        age2 = st.number_input('나이를 입력하세요', min_value=0, max_value=100, value=50, step=1, key='num_input2')
        gender2 = st.radio('성별은 선택하세요', ['male', 'female'], key='radio2')
    
else:   
    if st.button('폼 열기', key='btn_5'):
        st.session_state['clicked'] = True
        st.return()
        
            

Overwriting my_web/07_03.status.py
